In [13]:
import os
import pprint

from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Annotated

from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
model = init_chat_model("gpt-5-mini")

In [3]:
class Movie(BaseModel):
    title: str=Field("The title of the movie")
    genre: str=Field("The genre for the movie")
    year: int=Field("The year the movies was released")
    director: str=Field("The directory of the movie")
    length: int=Field("The lenght of the movie in minutes (rounded)")
    rating: float=Field("The movie's rating from 1-10")

In [4]:
new_flick = Movie(title="Star Wars", genre="Science Fiction", year=1975, director="Gearge Lucas", length=120)

In [5]:
print(new_flick)

title='Star Wars' genre='Science Fiction' year=1975 director='Gearge Lucas' length=120 rating="The movie's rating from 1-10"


In [6]:
model_with_structure = model.with_structured_output(Movie)

In [7]:
pprint.pprint(model_with_structure)

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, profile={'name': 'GPT-5 Mini', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000016A68A4D520>, async_client=<openai.resources.chat.completions

In [8]:
response = model_with_structure.invoke("Provide details about the movie 'The Bourne Identity'")

In [9]:
response

Movie(title='The Bourne Identity', genre='Action, Thriller', year=2002, director='Doug Liman', length=119, rating=7.9)

In [10]:
# Message output alongside parsed structure
class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movies was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

In [11]:
response = model_with_structure.invoke("Provide details about the movie 'Inception'")
response

{'raw': AIMessage(content='{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', additional_kwargs={'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 124, 'total_tokens': 222, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EANzuHyRiZ6FOFuXQMstxZ4WriM0R', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fde7e-1f8a-74d2-9003-138fced4d33c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 124, 'output_tokens': 98, 'total_tokens': 222, 'input_token_details': {'audio': 0, 'cache_re

In [12]:
# Nested structure

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide the details about the movie 'The Dark Knight Rises'")
response

MovieDetails(title='The Dark Knight Rises', year=2012, cast=[Actor(name='Christian Bale', role='Bruce Wayne / Batman'), Actor(name='Tom Hardy', role='Bane'), Actor(name='Anne Hathaway', role='Selina Kyle / Catwoman'), Actor(name='Michael Caine', role='Alfred Pennyworth'), Actor(name='Gary Oldman', role='Commissioner James Gordon'), Actor(name='Marion Cotillard', role='Miranda Tate / Talia al Ghul'), Actor(name='Joseph Gordon-Levitt', role='John Blake'), Actor(name='Morgan Freeman', role='Lucius Fox'), Actor(name='Cillian Murphy', role='Dr. Jonathan Crane / Scarecrow'), Actor(name='Juno Temple', role='Jen')], genres=['Action', 'Crime', 'Drama', 'Thriller'], budget=250.0)

# TypedDict
- a simpler alternative to Python's built-in typing, ideal when you don't need runtime validation

In [14]:
class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, ..., "The title of the movie."]
    year: Annotated[int, ..., "The year the movie was released."]
    director: Annotated[str, ..., "The directory of the movie."]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [20]:
model_with_td = model.with_structured_output(MovieDict)

In [21]:
response = model_with_td.invoke("Please provide the details of the movie Avengers End Game")

In [23]:
response

{'title': 'Avengers: Endgame',
 'year': 2019,
 'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4}

In [24]:
model.profile

{'name': 'GPT-5 Mini',
 'release_date': '2025-08-07',
 'last_updated': '2025-08-07',
 'open_weights': False,
 'max_input_tokens': 272000,
 'max_output_tokens': 128000,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': False,
 'image_url_inputs': True,
 'pdf_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True,
 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']}